# Domain-Specific RAG for Interview Preparation

This notebook shows how to build a Retrieval-Augmented Generation (RAG)
pipeline with Haystack for interview preparation. We create a small
domain-specific knowledge base with machine learning interview notes,
index it with FastEmbed, retrieve relevant passages, and use an LLM to
generate grounded answers.


## Install dependencies


In [6]:
%pip install fastembed-haystack qdrant-haystack
%pip install nltk>=3.9.1



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Create interview-preparation documents


In [7]:
from haystack.dataclasses import Document

raw_docs = [
    Document(
        content=(
            "Bias-variance tradeoff is the balance between underfitting and overfitting. "
            "High bias leads to overly simple models that miss patterns. "
            "High variance leads to models that fit noise in the training data."
        ),
        meta={"title": "Bias-Variance Tradeoff", "category": "ml_fundamentals"},
    ),
    Document(
        content=(
            "Precision measures how many predicted positive examples are actually positive. "
            "Recall measures how many actual positive examples were correctly identified. "
            "Precision is important when false positives are costly. "
            "Recall is important when false negatives are costly."
        ),
        meta={"title": "Precision and Recall", "category": "evaluation"},
    ),
    Document(
        content=(
            "Overfitting happens when a model learns noise and specific details from the training data "
            "instead of general patterns. Common mitigation strategies include regularization, "
            "cross-validation, simplifying the model, early stopping, and collecting more data."
        ),
        meta={"title": "Overfitting", "category": "modeling"},
    ),
    Document(
        content=(
            "In machine learning interviews, candidates are often asked to explain how they would "
            "deploy a model to production. A strong answer should mention model serving, monitoring, "
            "logging, latency, scalability, retraining, and rollback plans."
        ),
        meta={"title": "Model Deployment Interview Answer", "category": "ml_system_design"},
    ),
    Document(
        content=(
            "Behavioral interview answers are often structured with the STAR method: "
            "Situation, Task, Action, Result. This helps candidates give concise and evidence-based responses."
        ),
        meta={"title": "STAR Method", "category": "behavioral"},
    ),
]

len(raw_docs)


5

## Clean, split, and index documents in Qdrant


In [8]:
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack_integrations.components.embedders.fastembed import FastembedDocumentEmbedder
from haystack.document_stores.types import DuplicatePolicy


In [10]:
document_store = QdrantDocumentStore(
    ":memory:",
    embedding_dim=384,
    recreate_index=True,
    return_embedding=True,
    wait_result_from_api=True,
)


In [14]:
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter

cleaner = DocumentCleaner()
splitter = DocumentSplitter(split_by="period", split_length=3)

cleaned_docs = cleaner.run(raw_docs)["documents"]
split_docs = splitter.run(cleaned_docs)["documents"]

len(split_docs)

6

## Embed and write documents


In [15]:
document_embedder = FastembedDocumentEmbedder(
    model="BAAI/bge-small-en-v1.5",
    parallel=0,
    meta_fields_to_embed=["title", "category"],
)

documents_with_embeddings = document_embedder.run(split_docs)["documents"]
document_store.write_documents(documents_with_embeddings, policy=DuplicatePolicy.OVERWRITE)


Calculating embeddings: 100%|██████████| 6/6 [00:00<00:00, 98.06it/s]
100it [00:00, 50588.64it/s]          


6

## Build the RAG pipeline


In [16]:
from haystack import Pipeline
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever
from haystack_integrations.components.embedders.fastembed import FastembedTextEmbedder
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceAPIChatGenerator
from haystack.dataclasses import ChatMessage


In [18]:
from getpass import getpass
import os

os.environ["HF_API_TOKEN"] = getpass("Enter your Hugging Face token: ")



In [19]:
generator = HuggingFaceAPIChatGenerator(
    api_type="serverless_inference_api",
    api_params={"model": "Qwen/Qwen2.5-7B-Instruct", "provider": "together"},
    generation_kwargs={"max_tokens": 300},
)


In [20]:
template = [
    ChatMessage.from_user(
        """
You are an interview preparation assistant.
Answer the question using only the information contained in the documents.
If the answer cannot be inferred from the documents, say \"I don't know.\"

Documents:
{% for doc in documents %}
- {{ doc.content }}
{% endfor %}

Question: {{ question }}
Answer:
"""
    )
]

prompt_builder = ChatPromptBuilder(template=template)


ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


In [21]:
query_pipeline = Pipeline()
query_pipeline.add_component(
    "text_embedder",
    FastembedTextEmbedder(model="BAAI/bge-small-en-v1.5", parallel=0, prefix="query:"),
)
query_pipeline.add_component(
    "retriever",
    QdrantEmbeddingRetriever(document_store=document_store, top_k=3),
)
query_pipeline.add_component("prompt_builder", prompt_builder)
query_pipeline.add_component("generator", generator)

query_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
query_pipeline.connect("retriever.documents", "prompt_builder.documents")
query_pipeline.connect("prompt_builder", "generator")


🚅 Components
  - text_embedder: FastembedTextEmbedder
  - retriever: QdrantEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - generator: HuggingFaceAPIChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> generator.messages (list[ChatMessage])

## Ask interview-preparation questions


In [22]:
question = "How should I explain overfitting in an interview?"

results = query_pipeline.run(
    {
        "text_embedder": {"text": question},
        "prompt_builder": {"question": question},
    }
)

for reply in results["generator"]["replies"]:
    print(reply.text)


Calculating embeddings: 100%|██████████| 1/1 [00:00<00:00, 12.54it/s]


Overfitting happens when a model learns noise and specific details from the training data instead of general patterns. To mitigate overfitting, common strategies include regularization, cross-validation, simplifying the model, early stopping, and collecting more data.


In [23]:
question = "What is the STAR method?"

results = query_pipeline.run(
    {
        "text_embedder": {"text": question},
        "prompt_builder": {"question": question},
    }
)

for reply in results["generator"]["replies"]:
    print(reply.text)


Calculating embeddings: 100%|██████████| 1/1 [00:00<00:00, 25.82it/s]


The STAR method is a structured approach used in behavioral interview answers. It stands for Situation, Task, Action, and Result. This method helps candidates provide concise and evidence-based responses.
